# TreeMemory vs LoRA Benchmark

This notebook compares external hierarchical TreeMemory against LoRA fine-tuning for factual memory updates.

## Runtime

Use `Runtime -> Change runtime type -> T4 GPU` before running all cells.

In [ ]:
REPO_URL = "https://github.com/g1g4b1t/tree-memory.git"

import os
import shutil
import subprocess
from pathlib import Path

repo_dir = Path("/content/tree-memory")
if repo_dir.exists():
    shutil.rmtree(repo_dir)

subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
os.chdir(repo_dir)
print("Cloned to", Path.cwd())
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=False)

required_paths = [
    "benchmarks/lora_vs_tree_benchmark.py",
    "benchmarks/scaled_memory_benchmark.py",
    "requirements-lora.txt",
]
missing = [path for path in required_paths if not Path(path).exists()]
if missing:
    raise FileNotFoundError("Missing required files after clone: " + ", ".join(missing))
print("Required files found.")

## Install Dependencies

In [ ]:
import importlib.util
import subprocess
import sys

# Colab already includes torch for the selected runtime. Install only the missing LoRA stack.
needed = []
for package in ["pandas", "transformers", "peft", "sentencepiece"]:
    if importlib.util.find_spec(package) is None:
        needed.append(package)

if needed:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *needed], check=True)

import torch
print("Dependencies ready.")
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

## Run Quick LoRA vs TreeMemory Benchmark

Default model: `google/flan-t5-small`.

In [ ]:
import subprocess
import sys
from pathlib import Path

script = Path("benchmarks/lora_vs_tree_benchmark.py")
cmd = [
    sys.executable,
    str(script),
    "--model", "google/flan-t5-small",
    "--max-concepts", "16",
    "--max-eval-queries", "32",
    "--base-epochs", "4",
    "--update-epochs", "8",
    "--batch-size", "4",
    "--device", "auto",
]

print("Running:", " ".join(cmd))
proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(proc.stdout)
if proc.returncode != 0:
    raise RuntimeError(f"LoRA benchmark failed with exit code {proc.returncode}. Full log is printed above.")

## Inspect Results

In [ ]:
import json
import pandas as pd

with open("artifacts/results/lora_vs_tree_benchmark_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

print("LoRA train seconds:", results["base_train_seconds"])
print("LoRA update seconds:", results["update_train_seconds"])
print("Trainable LoRA params:", results["trainable_lora_params"])
pd.DataFrame(results["overall"])